# Audio Flamingo Next MP3 Sound Effects Listing

Upload an MP3 and generate a detailed plain-text inventory of sound effects heard with `nvidia/audio-flamingo-next-think-hf`.

Use a GPU runtime if possible. The model is an 8B BF16 checkpoint, so CPU-only inference is usually impractically slow. A 24 GB VRAM GPU is a practical minimum for short clips; 48 GB VRAM is safer for longer audio.

In [ ]:
# Run this once per fresh notebook runtime.
# Do not upgrade torch here; Colab/Kaggle images usually have a matching PyTorch/CUDA stack already.
# Remove torchvision if present. It is not used here, and a mismatched torchvision build can break transformers imports.
%pip install -q --upgrade pip
%pip uninstall -y -q torchvision
%pip install -q --upgrade transformers accelerate librosa soundfile

In [ ]:
import os
from contextlib import nullcontext
from pathlib import Path

import torch
from transformers import AutoConfig, AutoModel, AutoModelForSeq2SeqLM, AutoProcessor

MODEL_ID = os.environ.get("AFNEXT_MODEL_ID", "nvidia/audio-flamingo-next-think-hf")


def preferred_runtime() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


print(f"Model: {MODEL_ID}")
RUNTIME = preferred_runtime()
print(f"Runtime device: {RUNTIME}")
if RUNTIME == "cpu":
    print("Warning: no GPU detected. This model is large, and CPU inference can be very slow.")

In [ ]:
DTYPE = torch.bfloat16 if RUNTIME in {"cuda", "mps"} else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_ID)
config = AutoConfig.from_pretrained(MODEL_ID)

model_kwargs = {
    "torch_dtype": DTYPE,
    "low_cpu_mem_usage": True,
}

if RUNTIME == "cuda":
    model_kwargs["device_map"] = "auto"
elif RUNTIME == "mps":
    model_kwargs["device_map"] = {"": "mps"}

try:
    auto_model_cls = AutoModel._model_mapping[type(config)]
    auto_model_cls_name = auto_model_cls.__name__
except Exception as exc:
    auto_model_cls_name = f"unavailable ({type(exc).__name__}: {exc})"

print(f"AutoModel resolves to: {auto_model_cls_name}")
if str(auto_model_cls_name).endswith("ForConditionalGeneration"):
    model = AutoModel.from_pretrained(MODEL_ID, **model_kwargs).eval()
else:
    print("Using AutoModelForSeq2SeqLM because this Transformers install maps AutoModel to a non-generative class.")
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, **model_kwargs).eval()


def model_input_device(model) -> torch.device:
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cpu")


MODEL_DEVICE = model_input_device(model)
MODEL_DTYPE = next(model.parameters()).dtype
print(f"Loaded {MODEL_ID} on {MODEL_DEVICE} with dtype {MODEL_DTYPE}.")

## Upload Audio

In Google Colab, the next cell opens a file picker. In local Jupyter, paste the path to your MP3 or other audio file.

In [ ]:
def upload_or_choose_audio() -> str:
    try:
        from google.colab import files
    except ModuleNotFoundError:
        audio_path = input("Paste the path to your MP3/audio file: ").strip()
        if not audio_path:
            raise ValueError("No audio path provided.")
        return str(Path(audio_path).expanduser().resolve())

    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded.")
    first_name = next(iter(uploaded))
    return str(Path(first_name).resolve())


AUDIO_PATH = upload_or_choose_audio()
print(AUDIO_PATH)

## Sound Effects Prompt

In [ ]:
QUESTION = """
Analyze the audio as a sound-effects and environment-sound detector.

The audio contains only sound effects and environmental sounds. It should not contain music, speech, voices, singing, lyrics, dialogue, or narration, so do not add music or voice categories. Identify every distinct sound effect or environmental sound you can hear, including subtle background sounds and short foreground events.

Use this checklist to find sounds, but only output sounds that are actually present:
- Human/object movement: footsteps, running, shuffling, clothing rustle, hand contact, knocks, dragging, drops, impacts
- Vehicles: cars, trucks, buses, motorcycles, trains, aircraft, engines, horns, tires, brakes, sirens
- Nature/weather: wind, rain, thunder, water, waves, fire, leaves, trees, insects
- Animals: dogs, cats, birds, livestock, wildlife, animal movement
- Indoor objects: doors, windows, keys, dishes, appliances, electronics, tools, paper, furniture
- Outdoor/urban ambience: traffic rumble, crowd ambience, construction, machinery, alarms, electrical hum
- Impacts and Foley: hits, crashes, scrapes, squeaks, creaks, whooshes, swishes, transitions
- Other or uncertain sound effects: any non-music, non-voice sound that does not fit the categories above

Identify the relevant sound events first. Reason step by step with approximate timestamps or audio moments, and ground your explanation in what is heard at those moments. Use the checklist to reason about subtle background sounds, short foreground events, repeated events, and ambiguous sounds. It is okay to write this reasoning before the final answer.

After the reasoning, write a plain-text section titled "Sound effects heard:". List each distinct sound effect or environmental sound as a bullet with a concise label and a short description of the evidence. Examples: "distant traffic rumble - low continuous road noise in the background", "car driving by - engine and tire noise passing across the scene", "footsteps - repeated foot impacts", "wind - steady rushing air", "car horn - short honk", or "electrical hum - steady buzzing tone".

Deduplicate repeated instances of the same sound, but keep distinct sounds separate. If a sound is uncertain, include it with words like "possible" or "likely" instead of omitting it. Do not return JSON, an empty array, or only "no sounds" unless the clip is actually silent.
""".strip()

In [ ]:
MAX_NEW_TOKENS = 4096
REPETITION_PENALTY = 1.2

In [ ]:
def ask_audio(
    audio_path: str,
    question: str,
    max_new_tokens: int = 1024,
    repetition_penalty: float = 1.2,
) -> str:
    conversation = [
        [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": question.strip() or "Identify the relevant sound events first. Reason step by step with timestamps, then give a plain-text bullet list of sound effects heard."},
                    {"type": "audio", "path": str(audio_path)},
                ],
            }
        ]
    ]

    batch = processor.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
    )
    batch = batch.to(MODEL_DEVICE)

    if "input_features" in batch:
        batch["input_features"] = batch["input_features"].to(MODEL_DTYPE)

    use_cuda_amp = MODEL_DEVICE.type == "cuda" and MODEL_DTYPE in {torch.float16, torch.bfloat16}
    amp_context = torch.autocast("cuda", dtype=MODEL_DTYPE) if use_cuda_amp else nullcontext()

    with torch.inference_mode(), amp_context:
        generated = model.generate(
            **batch,
            max_new_tokens=int(max_new_tokens),
            repetition_penalty=float(repetition_penalty),
        )

    prompt_len = batch["input_ids"].shape[1]
    completion = generated[:, prompt_len:]
    return processor.batch_decode(
        completion,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()


answer = ask_audio(
    AUDIO_PATH,
    QUESTION,
    max_new_tokens=MAX_NEW_TOKENS,
    repetition_penalty=REPETITION_PENALTY,
)
print(answer)